# PSNR

Peak Signal-to-Noise Ratio. Mide la relación entre la señal máxima posible y el ruido introducido por la compresión. Se basa en el MSE entre imagen original y distorsionada. Valores más altos indican mejor calidad (típicamente 30–50 dB para buena calidad).

In [1]:
from skimage import io, color
from skimage.metrics import peak_signal_noise_ratio

def safe_rgb(img):
    if img.ndim == 2:  # grayscale
        return color.gray2rgb(img)
    if img.shape[-1] == 4:  # RGBA
        return color.rgba2rgb(img)
    return img

def calcular_psnr(img_orig_path, img_comp_path):
    orig = safe_rgb(io.imread(img_orig_path))
    comp = safe_rgb(io.imread(img_comp_path))
    
    orig = color.rgb2gray(orig)
    comp = color.rgb2gray(comp)
    
    return peak_signal_noise_ratio(orig, comp, data_range=1.0)

In [2]:
print(calcular_psnr('cute.png', 'imagenes/cute.jpg'))


46.3892106695685


# SSIM

Structural Similarity Index. Mide similitud entre dos imágenes comparando luminancia, contraste y estructura. Resultado entre -1 y 1; donde 1 = similitud perfecta. Mejor correlación con percepción humana que PSNR.

In [3]:
from skimage import io
from skimage.metrics import structural_similarity

def calcular_ssim(img_orig_path, img_comp_path):
    orig = safe_rgb(io.imread(img_orig_path))
    comp = safe_rgb(io.imread(img_comp_path))
    
    return structural_similarity(
        orig, comp,
        channel_axis=2,
        data_range=1.0
    )

In [4]:
score= calcular_ssim('cute.png', 'imagenes/cute.jpg')   
print(f'SSIM global: {score}')

SSIM global: 0.04886610868169924


# MS-SSIM

Multi-Scale Structural Similarity. Extensión de SSIM que evalúa similitud estructural en múltiples escalas espaciales (resoluciones). Captura degradaciones visuales tanto en detalles finos como en estructuras globales.

In [5]:
import torch
import torch.nn.functional as F
import numpy as np
from skimage import io, color
from pytorch_msssim import ms_ssim

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_rgb(img):
    if img.ndim == 2:  # grayscale
        return color.gray2rgb(img)
    if img.shape[-1] == 4:  # RGBA
        return color.rgba2rgb(img)
    return img

def load_tensor(path):
    img = safe_rgb(io.imread(path))
    img = img.astype(np.float32) / 255.0
    return torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = F.interpolate(
            img2,
            size=img1.shape[2:],
            mode='bilinear',
            align_corners=False
        )
    return img1, img2

# ---------------------------
# MS-SSIM
# ---------------------------

def calcular_ms_ssim(img_orig_path, img_comp_path):
    orig = load_tensor(img_orig_path)
    comp = load_tensor(img_comp_path)
    
    orig, comp = match_size(orig, comp)
    
    score = ms_ssim(orig, comp, data_range=1.0, size_average=True)
    return score.item()

In [6]:
print(calcular_ms_ssim('cute.png', 'imagenes/cute.jpg'))

0.18311285972595215


# LPIPS


Learned Perceptual Image Patch Similarity. Mide diferencia perceptual usando características extraídas de redes neuronales (VGG/AlexNet). Valores más bajos = más similares. Alta correlación con percepción humana.

In [7]:
import torch
import torch.nn.functional as F
import numpy as np
from skimage import io, color
import lpips

# ---------------------------
# CONFIGURACIÓN GLOBAL (evita recargar modelo cada vez)
# ---------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss_fn = lpips.LPIPS(net='vgg').to(device)
loss_fn.eval()

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_rgb(img):
    if img.ndim == 2:  # grayscale
        return color.gray2rgb(img)
    if img.shape[-1] == 4:  # RGBA
        return color.rgba2rgb(img)
    return img

def load_tensor(path):
    img = safe_rgb(io.imread(path))
    img = img.astype(np.float32) / 127.5 - 1.0  # [-1, 1]
    return torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = F.interpolate(
            img2,
            size=img1.shape[2:],
            mode='bilinear',
            align_corners=False
        )
    return img1, img2

# ---------------------------
# LPIPS
# ---------------------------

def calcular_lpips(img_orig_path, img_comp_path):
    orig = load_tensor(img_orig_path)
    comp = load_tensor(img_comp_path)
    
    orig, comp = match_size(orig, comp)
    
    with torch.no_grad():
        dist = loss_fn(orig, comp)
    
    return dist.item()

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\lpips\weights\v0.1\vgg.pth


In [8]:
print(calcular_lpips('cute.png', 'imagenes/cute.jpg'))

0.6470681428909302


# VIF

Visual Information Fidelity. Mide cuánta información visual del original se conserva en la imagen distorsionada, basándose en teoría de la información y modelos del sistema visual humano.

In [9]:
import numpy as np
from skimage import io, color
from sewar.full_ref import vifp
from skimage.transform import resize

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_gray(img):
    if img.ndim == 3:
        if img.shape[-1] == 4:
            img = color.rgba2rgb(img)
        img = color.rgb2gray(img)
    return img

def load_gray(path):
    img = io.imread(path)
    img = safe_gray(img)
    return img.astype(np.float64)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = resize(img2, img1.shape, anti_aliasing=True)
    return img1, img2

# ---------------------------
# VIF (REAL)
# ---------------------------

def calcular_vif(img_orig_path, img_comp_path):
    orig = load_gray(img_orig_path)
    comp = load_gray(img_comp_path)
    
    orig, comp = match_size(orig, comp)
    
    return vifp(orig, comp)

In [10]:
print(calcular_vif('cute.png', 'imagenes/cute.jpg'))

0.9944796332450144


# FSIM

Feature Similarity Index. Evalúa similitud perceptual usando congruencia de fase (Phase Congruency) y magnitud del gradiente. Muy efectivo para detectar degradaciones en bordes, contornos y texturas.

In [11]:
import numpy as np
from skimage import io, color
from scipy.ndimage import sobel
from skimage.transform import resize

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_gray(img):
    if img.ndim == 3:
        if img.shape[-1] == 4:
            img = color.rgba2rgb(img)
        img = color.rgb2gray(img)
    return img

def load_gray(path):
    img = io.imread(path)
    img = safe_gray(img)
    return img.astype(np.float64)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = resize(img2, img1.shape, anti_aliasing=True)
    return img1, img2

# ---------------------------
# FSIM APROXIMADO
# ---------------------------

def calcular_fsim_aproximado(img_orig_path, img_comp_path, T1=0.85, T2=160):
    orig = load_gray(img_orig_path)
    comp = load_gray(img_comp_path)
    
    orig, comp = match_size(orig, comp)

    def grad_mag(img):
        gx = sobel(img, axis=1)
        gy = sobel(img, axis=0)
        return np.sqrt(gx**2 + gy**2)

    G1 = grad_mag(orig)
    G2 = grad_mag(comp)

    # Similaridad de gradiente
    GMS = (2 * G1 * G2 + T2) / (G1**2 + G2**2 + T2)

    # Similaridad de luminancia
    SL = (2 * orig * comp + T1) / (orig**2 + comp**2 + T1)

    return np.mean(GMS * SL)

In [12]:
print(calcular_fsim_aproximado('cute.png', 'imagenes/cute.jpg'))

0.9999810116784077


# GMSD

Gradient Magnitude Similarity Deviation. Compara la magnitud de gradientes entre imagen original y distorsionada. Bajo costo computacional y buena correlación perceptual. Valores más bajos = mejor calidad.

In [13]:
import numpy as np
from skimage import io, color
from scipy.ndimage import sobel
from skimage.transform import resize

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_gray(img):
    if img.ndim == 3:
        if img.shape[-1] == 4:
            img = color.rgba2rgb(img)
        img = color.rgb2gray(img)
    return img

def load_gray(path):
    img = io.imread(path)
    img = safe_gray(img)
    return img.astype(np.float64)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = resize(img2, img1.shape, anti_aliasing=True)
    return img1, img2

# ---------------------------
# GMSD
# ---------------------------

def calcular_gmsd(img_orig_path, img_comp_path, C=0.0026):
    orig = load_gray(img_orig_path)
    comp = load_gray(img_comp_path)
    
    orig, comp = match_size(orig, comp)

    def grad_mag(arr):
        gx = sobel(arr, axis=1)
        gy = sobel(arr, axis=0)
        return np.sqrt(gx**2 + gy**2)

    Gx = grad_mag(orig)
    Gy = grad_mag(comp)

    # Similaridad de gradiente
    GMS = (2 * Gx * Gy + C) / (Gx**2 + Gy**2 + C)

    # Desviación estándar del mapa
    return np.std(GMS)

In [14]:
print(calcular_gmsd('cute.png', 'imagenes/cute.jpg'))

0.06317453712437866


# MS-SSIM + LPIPS (combinado)

Enfoque híbrido que combina MS-SSIM (similitud estructural multiescala) y LPIPS (diferencia perceptual profunda). Captura simultáneamente distorsiones estructurales globales y diferencias perceptuales de alto nivel.

In [25]:
import torch
import torch.nn.functional as F
import numpy as np
from skimage import io, color
import lpips
from pytorch_msssim import ms_ssim

# ---------------------------
# CONFIGURACIÓN GLOBAL
# ---------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss_fn = lpips.LPIPS(net='vgg').to(device)
loss_fn.eval()

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_rgb(img):
    if img.ndim == 2:
        return color.gray2rgb(img)
    if img.shape[-1] == 4:
        return color.rgba2rgb(img)
    return img

def load_np(path):
    img = safe_rgb(io.imread(path))
    return img.astype(np.float32)

def match_size_np(img1, img2):
    if img1.shape != img2.shape:
        from skimage.transform import resize
        img2 = resize(img2, img1.shape, anti_aliasing=True)
    return img1, img2

# ---------------------------
# MÉTRICA HÍBRIDA
# ---------------------------

def calcular_ms_ssim_lpips(img_orig_path, img_comp_path, alpha=0.5, beta=0.5):
    
    arr_o = load_np(img_orig_path)
    arr_c = load_np(img_comp_path)

    arr_o, arr_c = match_size_np(arr_o, arr_c)

    # ---- MS-SSIM [0,1]
    t_o = torch.from_numpy(arr_o / 255.0).permute(2,0,1).unsqueeze(0).to(device)
    t_c = torch.from_numpy(arr_c / 255.0).permute(2,0,1).unsqueeze(0).to(device)

    ms = ms_ssim(t_o, t_c, data_range=1.0, size_average=True).item()

    # ---- LPIPS [0,+inf] (menor mejor)
    lp_o = torch.from_numpy(arr_o / 127.5 - 1).permute(2,0,1).unsqueeze(0).to(device)
    lp_c = torch.from_numpy(arr_c / 127.5 - 1).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        lp = loss_fn(lp_o, lp_c).item()

    # ---- Score combinado (mayor = mejor)
    score = alpha * ms + beta * (1 - lp)

    return {
        'ms_ssim': ms,
        'lpips': lp,
        'combinado': score
    }

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\lpips\weights\v0.1\vgg.pth


In [26]:
reultado = calcular_ms_ssim_lpips('cute.png', 'imagenes/cute.jpg') 

print(f"Combinado: {reultado['combinado']:.4f}")


Combinado: 0.2680


In [27]:
import warnings
import torch
import numpy as np
from skimage import io, color
from skimage.transform import resize
import lpips
from pytorch_msssim import ms_ssim

# ---------------------------
# CONFIGURACIÓN GLOBAL
# ---------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Carga única del modelo — costoso, no repetir por llamada
loss_fn = lpips.LPIPS(net='vgg').to(device)
loss_fn.eval()
for p in loss_fn.parameters():
    p.requires_grad_(False)   # evita acumulación de grafo innecesaria

LPIPS_MIN_SIZE = 64  # mínimo requerido por VGG interno de LPIPS

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_rgb(img: np.ndarray) -> np.ndarray:
    """Garantiza salida HxWx3 uint8 o float32 según entrada."""
    if img.ndim == 2:
        img = color.gray2rgb(img)
    elif img.shape[-1] == 4:
        img = color.rgba2rgb(img)      # rgba2rgb ya devuelve [0,1] float
    return img

def load_np(path: str) -> np.ndarray:
    """
    Carga imagen y la devuelve en float32 con rango [0, 255].
    Detecta imágenes ya normalizadas a [0,1] y las reescala.
    """
    img = safe_rgb(io.imread(path))

    # rgba2rgb de skimage devuelve float en [0,1]; resto viene en [0,255] uint
    if img.dtype in (np.float32, np.float64) and img.max() <= 1.0:
        img = (img * 255.0)

    return img.astype(np.float32)

def match_size_np(
    img1: np.ndarray,
    img2: np.ndarray,
    silent: bool = False
) -> tuple[np.ndarray, np.ndarray]:
    """
    Redimensiona img2 al tamaño de img1 si difieren.
    Emite advertencia por defecto — nunca falla silenciosamente.
    """
    if img1.shape != img2.shape:
        if not silent:
            warnings.warn(
                f"Tamaños distintos: {img1.shape} vs {img2.shape}. "
                "Se redimensiona la imagen comprimida — esto puede afectar la métrica.",
                UserWarning,
                stacklevel=2,
            )
        img2 = (resize(img2, img1.shape, anti_aliasing=True) * 255.0).astype(np.float32)
    return img1, img2

def _validate_size(arr: np.ndarray, name: str) -> None:
    h, w = arr.shape[:2]
    if h < LPIPS_MIN_SIZE or w < LPIPS_MIN_SIZE:
        raise ValueError(
            f"{name} demasiado pequeña ({h}×{w}). "
            f"LPIPS/VGG requiere mínimo {LPIPS_MIN_SIZE}×{LPIPS_MIN_SIZE} px."
        )

# ---------------------------
# NORMALIZACIÓN LPIPS
# ---------------------------

def _clamp_lpips(lp_raw: float, percentile_cap: float = 1.0) -> float:
    """
    LPIPS no está acotado en [0,1].
    Se normaliza con un cap configurable para mantener el score combinado en [0,1].
    Cap por defecto = 1.0 (distorsión máxima esperada en la mayoría de casos).
    Valores sobre el cap se truncan a 0 en la componente (1 - lp).
    """
    return min(lp_raw, percentile_cap) / percentile_cap  # → [0,1]

# ---------------------------
# MÉTRICA HÍBRIDA
# ---------------------------

def calcular_ms_ssim_lpips(
    img_orig_path: str,
    img_comp_path: str,
    alpha: float = 0.5,
    beta: float = 0.5,
    lpips_cap: float = 1.0,
    silent_resize: bool = False,
) -> dict:
    """
    Métrica híbrida MS-SSIM + LPIPS.

    Args:
        img_orig_path:  Ruta imagen de referencia.
        img_comp_path:  Ruta imagen comprimida/transformada.
        alpha:          Peso de MS-SSIM en el score combinado.
        beta:           Peso de LPIPS en el score combinado.
        lpips_cap:      Valor máximo de LPIPS antes de normalizar a [0,1].
                        Ajustar según el dominio de transformaciones.
        silent_resize:  Si True, no emite advertencia al redimensionar.

    Returns:
        dict con ms_ssim, lpips_raw, lpips_norm y combinado (todos en [0,1]).
    """
    if not (0.0 <= alpha <= 1.0 and 0.0 <= beta <= 1.0):
        raise ValueError("alpha y beta deben estar en [0, 1].")

    arr_o = load_np(img_orig_path)
    arr_c = load_np(img_comp_path)
    arr_o, arr_c = match_size_np(arr_o, arr_c, silent=silent_resize)

    _validate_size(arr_o, "Imagen original")
    _validate_size(arr_c, "Imagen comprimida")

    with torch.no_grad():

        # ---- MS-SSIM — entrada en [0, 1]
        t_o = torch.from_numpy(arr_o / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)
        t_c = torch.from_numpy(arr_c / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)

        ms = ms_ssim(t_o, t_c, data_range=1.0, size_average=True).item()

        # ---- LPIPS — entrada en [-1, 1]
        lp_o = (t_o * 2.0) - 1.0
        lp_c = (t_c * 2.0) - 1.0

        lp_raw = loss_fn(lp_o, lp_c).item()

    lp_norm = 1.0 - _clamp_lpips(lp_raw, lpips_cap)  # mayor = mejor, en [0,1]

    score = alpha * ms + beta * lp_norm

    return {
        'ms_ssim':    round(ms, 6),
        'lpips_raw':  round(lp_raw, 6),   # valor original sin acotar
        'lpips_norm': round(lp_norm, 6),  # normalizado a [0,1]
        'combinado':  round(score, 6),    # siempre en [0,1]
    }

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\lpips\weights\v0.1\vgg.pth


In [30]:
resultados = calcular_ms_ssim_lpips('cute.png', 'imagenes/cute.jpg') 

print(f"Combinado: {resultados['lpips_raw']:.4f}")

Combinado: 0.0280
